<a href="https://colab.research.google.com/github/Harishse616/AI/blob/main/AI_FT_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### installing the dependencies to modify an llm


In [ ]:
%%capture
!pip install unsloth

### importing the llm from unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

### adding loRA to it

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

Unsloth 2026.6.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import Dataset

data = [
    {
        "text":
        "### Instruction:\nBuild a language learning platform\n\n### Response:\nStart with requirements analysis. Django apps: users, courses, lessons, progress. Database tables: User, Course, Lesson, Enrollment, Progress. Create REST APIs before frontend implementation."
    },
    {
        "text":
        "### Instruction:\nDesign a SaaS dashboard\n\n### Response:\nLayout structure: sidebar navigation, top navbar, KPI cards, analytics section, recent activity section. Prioritize usability and visual hierarchy."
    },
]


dataset = Dataset.from_list(data)

In [ ]:
import transformers
import trl
import unsloth

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)

try:
    print("Unsloth:", unsloth.__version__)
except:
    print("Can't get Unsloth version")


Transformers: 5.5.0
TRL: 0.24.0
Unsloth: 2026.6.2


In [ ]:
print(tokenizer)
print(type(tokenizer))

Qwen2Tokenizer(name_or_path='unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit', vocab_size=151643, model_max_length=32768, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|PAD_TOKEN|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_en

In [ ]:
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir="outputs",
    max_steps=20,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=config,
)

trainer.train()

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 20 | Total steps = 20
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.555551
2,3.555551
3,3.438238
4,3.059167
5,2.680060
6,2.323084
7,1.990489
8,1.657678
9,1.318728
10,1.028754


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-20/tokenizer_config.json.


TrainOutput(global_step=20, training_loss=1.3880907535552978, metrics={'train_runtime': 39.3512, 'train_samples_per_second': 4.066, 'train_steps_per_second': 0.508, 'total_flos': 74238728417280.0, 'train_loss': 1.3880907535552978, 'epoch': 20.0})

In [ ]:
inputs = tokenizer(
    "Create a Django user model",
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0]))


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Create a Django user model

To create a custom user model in Django, you need to follow these steps:

1. Create a new app if it doesn't already exist.
2. Define your custom user model by extending `AbstractUser`.
3. Update the settings to use your custom user model.

Here's an example of how to create a custom user model called `CustomUser`:

```python
# myapp/models.py

from django.contrib.auth.models import AbstractUser
from django.db import models

class CustomUser


In [ ]:
model.save_pretrained("t_lora")
tokenizer.save_pretrained("t_lora")

Unsloth: Restored added_tokens_decoder metadata in t_lora/tokenizer_config.json.


('t_lora/tokenizer_config.json',
 't_lora/chat_template.jinja',
 't_lora/tokenizer.json')

In [ ]:
!ls t_lora

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json
